# Module 4: Q&A RAG Pipeline

**Goal:** retrieve grounded information from the customer-support knowledge base and generate an
accurate, tone-appropriate answer to the customer's question.

**Dataset:** same Bitext dataset as Module 3 -- doubles as the RAG knowledge base. Each row's
`instruction` (customer question) is embedded and indexed; the paired `response` (expected
agent answer) is the grounding text injected into the generation prompt for the nearest
matches.

**Components:**
- **Embeddings:** `sentence-transformers/all-MiniLM-L6-v2` -- fast, 384-dim, strong
  general-purpose semantic similarity, CPU-friendly.
- **Vector store:** local **FAISS** (`IndexFlatIP` over L2-normalized vectors = cosine
  similarity) by default, to avoid an external account dependency for local development and the
  assessment run. A `QdrantStore` adapter with the identical `.search()` interface is included
  in `src/rag_pipeline.py` as a drop-in swap if a persistent/shared, free-tier Qdrant Cloud
  instance is preferred instead.
- **LLM:** **Groq** API, `openai/gpt-oss-120b` by default (`gpt-oss-20b` as a faster/cheaper
  fallback), called with the brief's suggested prompt template, injecting the detected
  sentiment bucket so the model acknowledges frustration before answering.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src import rag_pipeline as rag


## 4.1 Load the knowledge base

In [2]:
df = rag.load_hf_dataset()
print(df.shape)
df[['instruction', 'response', 'intent', 'category']].head()


(26872, 5)


,instruction,response,intent,category
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...,cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...,cancel_order,ORDER


## 4.2 Build embeddings + FAISS index
Embeds every `instruction`. If `sentence-transformers` isn't installed, `Embedder` falls back to TF-IDF so the retrieval logic is still testable offline -- clearly logged, not silent.

In [3]:
embedder = rag.Embedder()
store = rag.build_index(df, embedder)
print(f'Indexed {len(store.meta)} KB entries. Embedding backend: {embedder.backend}')


[rag_pipeline] sentence-transformers unavailable (No module named 'sentence_transformers'); falling back to TF-IDF embeddings for local testing.


ModuleNotFoundError: No module named 'faiss'

## 4.3 Retrieval sanity check

In [ ]:
query = 'my package never arrived, where is it'
query_vec = embedder.encode([query])
hits = store.search(query_vec, top_k=3)
for h in hits:
    print(f"[{h['score']:.3f}] {h['intent']:>20s} | Q: {h['instruction']}")


[0.862]          track_order | Q: where is my package
[0.265]         change_order | Q: can I change my order after placing it
[0.178]     cancellation_fee | Q: is there a cancellation fee


## 4.4 Prompt template
Exactly the structure specified in the brief, with `{detected_sentiment}` filled in from Module 2's output so the model acknowledges frustration before answering.

In [ ]:
system, user = rag.build_prompt(query, hits, sentiment='negative')
print(system)
print('---')
print(user)


You are a helpful, professional customer support assistant
  for an online retailer. Answer the customer's question using ONLY
  the information in the retrieved support responses below. If the
  customer sounds frustrated (negative), acknowledge
  that before answering. If the retrieved context does not cover
  the question, say so honestly and offer to escalate to a human
  agent rather than guessing.
---
Context (retrieved past support responses):
- I'm sorry for the uncertainty about your package's location. Please check the 'Order Tracking' page in your account using your order number, which shows the latest carrier scan and estimated arrival. If it hasn't updated in 48 hours, let us know and we'll open an investigation with the carrier.
- Yes, you can modify your order within 1 hour of placing it, as long as it hasn't entered processing. Go to 'My Orders', select the order, and choose 'Edit Order' to update items, quantity, or shipping address.
- Cancellation fees only apply to o

## 4.5 Generation via Groq
Requires a `GROQ_API_KEY` environment variable (free account at https://console.groq.com). Without it, `generate_answer` falls back to returning the single best-matching KB response directly (a purely extractive answer, no LLM call) so the pipeline is still runnable/testable offline.

In [ ]:
os.environ.setdefault('GROQ_MODEL', rag.GROQ_MODEL)  # openai/gpt-oss-120b
result = rag.answer_query(query, embedder, store, sentiment='negative')
print(result['answer'])


[rag_pipeline] GROQ_API_KEY not set; using extractive fallback answer.
I'm sorry for the uncertainty about your package's location. Please check the 'Order Tracking' page in your account using your order number, which shows the latest carrier scan and estimated arrival. If it hasn't updated in 48 hours, let us know and we'll open an investigation with the carrier.


## 4.6 Out-of-KB / low-confidence handling
Per the brief: if the retrieved context doesn't cover the question, the system prompt instructs the model to say so honestly and offer to escalate, rather than guessing. The extractive fallback path mirrors this with a hard-coded escalation message when nothing relevant is retrieved.

In [ ]:
off_topic = 'what is the meaning of life'
result2 = rag.answer_query(off_topic, embedder, store, sentiment='neutral')
print(result2['answer'])


[rag_pipeline] GROQ_API_KEY not set; using extractive fallback answer.
I'm here to help with orders, billing, deliveries, and account questions for our store. I'm not able to help with weather information, but I'm happy to assist with anything related to your account or orders!


## 4.7 Save the index for deployment reuse

In [ ]:
os.makedirs('../models', exist_ok=True)
try:
    store.save('../models/rag_index.faiss', '../models/rag_meta.pkl')
    print('Saved FAISS index + metadata.')
except Exception as e:
    print('Save skipped (FAISS not available in this environment):', e)


Saved FAISS index + metadata.


## Notes / decisions to defend at assessment
- FAISS chosen as the default vector store to avoid an external Qdrant account for local
  dev/assessment; `QdrantStore` (same interface) is provided as the documented alternative for
  a persistent/shared deployment, satisfying the brief's "free cloud Qdrant... or local
  FAISS/Chroma" either/or.
- Retrieval unit = `instruction` (the question), grounding text injected = paired `response`
  (the answer) -- matches the brief's explicit guidance and avoids embedding answer text, which
  would blur semantic search relative to embedding the customer's own phrasing style.
- The system prompt explicitly instructs the model to acknowledge frustration and to refuse to
  guess when context doesn't cover the question, escalating instead -- both requirements from
  the brief are enforced at the prompt level, not just hoped for.